In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, chi2



#Carga de datos

train_data = pd.read_csv("../../Data/train_indexado.csv")
test_data = pd.read_csv("../../Data/test_indexado.csv")

# Definir las clases de emociones
emotion_classes = train_data.columns[2:].tolist()




In [2]:
import re
from nltk.stem import WordNetLemmatizer

# --- PREPROCESAMIENTO PARA LEMATIZACIÓN ---
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = re.findall(r'\b\w+\b', text.lower())
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

X_train_lem = train_data['Text'].apply(preprocess_text)
X_test_lem = test_data['Text'].apply(preprocess_text)

In [3]:
# TF-IDF VECTORIZACIÓN
vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", max_features=10000)
X_train = vectorizer.fit_transform(X_train_lem)
X_test = vectorizer.transform(X_test_lem)
y_train = np.asarray(train_data[emotion_classes])
y_test = np.asarray(test_data[emotion_classes])

In [4]:
for k in [500, 1000, 2000, 3000, 5000]:
    print(f"\n*** Evaluación con Chi-cuadrada (Número de atributos: {k} ) ***")

    #Selección de features/características
    selector = SelectKBest(score_func=chi2, k=k)
    X_train_chi = selector.fit_transform(X_train, y_train)
    X_test_chi = selector.transform(X_test)

    #Perceptrón Multicapa (Red Neuronal)

    mlp_clf = MLPClassifier(max_iter=1000)
    multi_mlp = MultiOutputClassifier(mlp_clf)
    multi_mlp.fit(X_train_chi, y_train)

    y_pred_mlp = multi_mlp.predict(X_test_chi)

        #Reporte MLP
    report_dict_mlp = classification_report(y_test, y_pred_mlp, output_dict=True, zero_division=0)

    f1_macro_mlp = report_dict_mlp["macro avg"]["f1-score"]
    recall_macro_mlp = report_dict_mlp["macro avg"]["recall"]

    print("\nResultados Neural Network Multilabel:\n")
    print("Accuracy:", accuracy_score(y_test, y_pred_mlp))
    print(f"F1 Score (macro avg): {f1_macro_mlp:.5f}")
    print(f"Recall Score (macro avg): {recall_macro_mlp:.5f}")

    







*** Evaluación con Chi-cuadrada (Número de atributos: 500 ) ***

Resultados Neural Network Multilabel:

Accuracy: 0.3373871383821633
F1 Score (macro avg): 0.34525
Recall Score (macro avg): 0.29725

*** Evaluación con Chi-cuadrada (Número de atributos: 1000 ) ***

Resultados Neural Network Multilabel:

Accuracy: 0.3329648056016215
F1 Score (macro avg): 0.36625
Recall Score (macro avg): 0.31888

*** Evaluación con Chi-cuadrada (Número de atributos: 2000 ) ***

Resultados Neural Network Multilabel:

Accuracy: 0.32688409802837665
F1 Score (macro avg): 0.35144
Recall Score (macro avg): 0.30630

*** Evaluación con Chi-cuadrada (Número de atributos: 3000 ) ***

Resultados Neural Network Multilabel:

Accuracy: 0.33554449972360423
F1 Score (macro avg): 0.35223
Recall Score (macro avg): 0.30401

*** Evaluación con Chi-cuadrada (Número de atributos: 5000 ) ***

Resultados Neural Network Multilabel:

Accuracy: 0.32688409802837665
F1 Score (macro avg): 0.35049
Recall Score (macro avg): 0.30113
